# Bonus : export ONNX, quantification et démo

L'énoncé propose plusieurs bonus. Nous en faisons deux :

- **export ONNX et quantification**, avec la mesure du gain de taille et de latence ;
- **une petite application Gradio** : une photo en entrée, la maladie ou « inconnu » en sortie, avec la carte Grad-CAM

Le modèle utilisé est le classifieur **sans M** de la partie 4 (MobileNetV3-Large, 9 classes), associé à l'auto-encodeur des classes connues et à la règle OU.

- On ajoute le dossier du projet au chemin Python pour importer notre code (`src/`).
- `config` contient les chemins, la classe M choisie et le GPU utilisé.
- `set_seed(42)` fixe le hasard : on obtient les mêmes résultats à chaque exécution.
- La dernière ligne affiche la machine utilisée (à noter dans le rapport).

In [7]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # dossier du projet, pour pouvoir importer src/

from src import config as C
C.set_seed(42)
print(C.describe_env())

Python 3.12.10, torch 2.6.0+cu124, torchvision 0.21.0+cu124
Calcul sur : cuda
GPU : NVIDIA GeForce RTX 4080 Laptop GPU
CPU : 13th Gen Intel(R) Core(TM) i7-13650HX (20 coeurs logiques)


- On recharge le classifieur sans M **sur le CPU**, car un téléphone d'entrée de gamme n'a pas de GPU.
- On cache les messages d'avertissement trop bavards d'ONNX.
- Les fichiers ONNX sont rangés dans `results/onnx/` (non versionné, trop lourd).

In [8]:
import time
import logging
import warnings
import numpy as np
import pandas as pd
import torch
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, quantize_static, QuantType, QuantFormat, CalibrationDataReader
from onnxruntime.quantization.shape_inference import quant_pre_process

from src import data as D
from src.models import load_classifier
from src.metrics import upsert_summary

logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

clf, meta = load_classifier(C.CKPT_DIR / "clf_sans_M.pt", torch.device("cpu"))
SIZE = meta["size"]
NORM = meta["norm"]
CONNUES = meta["classes"]
DOSSIER = C.RESULTS / "onnx"
DOSSIER.mkdir(exist_ok=True)
df = D.load_splits()
print("Classifieur chargé sur le CPU :", len(CONNUES), "classes, images", SIZE, "x", SIZE)

Classifieur chargé sur le CPU : 9 classes, images 224 x 224


- `torch.onnx.export` enregistre le modèle au format ONNX, un format standard qu'on peut lire sur Android ; `check_model` vérifie le fichier.
- `quant_pre_process` prépare le fichier comme le conseille ONNX Runtime (fusion de couches, tailles connues à chaque étape).
- **Quantification dynamique** : seuls les poids passent de 32 bits (float) à 8 bits (entiers), les calculs intermédiaires restent en float. On ne l'applique qu'aux couches denses de la tête (`Gemm`) : sur les convolutions, ONNX Runtime la fait avec un opérateur lent et très imprécis (un premier essai sur tout le réseau donnait 44,6 % d'accuracy et une latence 5 fois plus grande).
- **Quantification statique** : les poids **et** les valeurs intermédiaires passent en 8 bits. Pour connaître la plage de ces valeurs, on fait passer 200 images du **train** dans le modèle (jamais le test).
- On affiche la taille des 3 fichiers. On s'attend à une division par environ 4 (32 bits → 8 bits).

In [9]:
f32 = DOSSIER / "clf_fp32.onnx"
torch.onnx.export(clf.eval(), torch.randn(1, 3, SIZE, SIZE), f32,
                  input_names=["image"], output_names=["logits"],
                  dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
                  opset_version=17, dynamo=False)
onnx.checker.check_model(onnx.load(f32))

# préparation conseillée par ONNX Runtime avant de quantifier (fusion de couches, formes connues)
f_pre = DOSSIER / "clf_fp32_prepare.onnx"
quant_pre_process(f32, f_pre)

# dynamique : seules les couches denses de la tête (Gemm) sont quantifiées
f_dyn = DOSSIER / "clf_int8_dynamique.onnx"
quantize_dynamic(f_pre, f_dyn, weight_type=QuantType.QInt8, op_types_to_quantize=["MatMul", "Gemm"])


class ImagesCalibration(CalibrationDataReader):
    """Donne 200 images du train, une par une, pour mesurer la plage des valeurs intermédiaires."""

    def __init__(self, n=200):
        train = D.subset(df, split="train", classes=CONNUES)
        train = train.sample(n, random_state=42)
        self.ds = D.LeafDataset(train, SIZE, D.get_transform("clf", False, NORM), CONNUES, preload=False)
        self.position = 0

    def get_next(self):
        if self.position >= len(self.ds):
            return None
        x = self.ds[self.position][0].unsqueeze(0).numpy()
        self.position += 1
        return {"image": x}


f_sta = DOSSIER / "clf_int8_statique.onnx"
quantize_static(f_pre, f_sta, ImagesCalibration(), quant_format=QuantFormat.QDQ, per_channel=True,
                activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)

for f in [f32, f_dyn, f_sta]:
    print(f"{f.name:28s} {f.stat().st_size / 1e6:.2f} Mo")

clf_fp32.onnx                16.85 Mo
clf_int8_dynamique.onnx      13.15 Mo
clf_int8_statique.onnx       4.73 Mo


- On prépare les images de test des 9 classes connues.
- `mesurer` ouvre le modèle avec ONNX Runtime sur **un seul thread CPU** (comme un petit téléphone), fait 20 passages de chauffe, mesure le temps sur 100 images, puis prédit tout le test.
- Tableau : taille, latence et accuracy pour les 3 versions. Les lignes vont aussi dans `summary.csv`.

In [10]:
test = D.subset(df, split="test", classes=CONNUES)
ds_test = D.LeafDataset(test, SIZE, D.get_transform("clf", False, NORM), CONNUES)
liste = []
for i in range(len(ds_test)):
    liste.append(ds_test[i][0].numpy())
X_test = np.stack(liste).astype(np.float32)
y_test = np.array(ds_test.labels)


def mesurer(fichier):
    """Latence moyenne et écart-type (ms) pour une image, sur 1 thread CPU, puis prédictions sur tout le test."""
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    session = ort.InferenceSession(str(fichier), options, providers=["CPUExecutionProvider"])
    for _ in range(20):                                   # chauffe
        session.run(None, {"image": X_test[:1]})
    temps = []
    for k in range(100):                                  # 100 images différentes
        image = X_test[k % len(X_test)][None]
        debut = time.perf_counter()
        session.run(None, {"image": image})
        temps.append((time.perf_counter() - debut) * 1000)
    predictions = []
    for i in range(0, len(X_test), 64):
        sorties = session.run(None, {"image": X_test[i:i + 64]})[0]
        predictions.append(sorties.argmax(axis=1))
    return np.mean(temps), np.std(temps), np.concatenate(predictions)


versions = {"float32": f32, "int8 dynamique": f_dyn, "int8 statique": f_sta}
lignes = {}
for nom in versions:
    fichier = versions[nom]
    moyenne, ecart, pred = mesurer(fichier)
    taille = fichier.stat().st_size / 1e6
    acc = (pred == y_test).mean()
    lignes[nom] = {"taille (Mo)": taille, "latence CPU (ms)": moyenne, "écart-type (ms)": ecart,
                   "accuracy (connues, test)": acc}
    upsert_summary({"experiment": f"Bonus_ONNX_{nom}", "partie": "bonus", "modele": f"MobileNetV3 sans M, ONNX {nom}",
                    "resolution": SIZE, "taille_mo": round(taille, 2), "latence_cpu_ms": round(moyenne, 2),
                    "test_acc": round(acc, 4)})

print("CPU :", C.cpu_name(), "| 1 thread")
pd.DataFrame(lignes).T.round(3)

CPU : 13th Gen Intel(R) Core(TM) i7-13650HX | 1 thread


,taille (Mo),latence CPU (ms),écart-type (ms),"accuracy (connues, test)"
float32,16.846,7.471,2.809,0.993
int8 dynamique,13.147,5.465,2.192,0.993
int8 statique,4.728,5.286,2.448,0.882


### Ce qu'on observe

*(à écrire avec nos résultats)*

## Démo Gradio

`app.py` (à la racine du projet) charge le classifieur sans M, l'auto-encodeur des classes connues (`ae_connus.pt`) et les seuils de la règle OU (`hybrid_config.json`), tous produits par le notebook 04. On peut la lancer ici ou avec `python app.py`.

- On teste la démo sans interface sur 3 images : une maladie connue du test, une image de la maladie inconnue M, et une de nos photos de mains.
- Pour chaque image, on affiche la réponse du système : la maladie avec sa confiance, ou « INCONNU » avec la raison du rejet.
- Ce sont les 3 cas à montrer pendant la soutenance.

In [11]:
import importlib
from PIL import Image
import app
importlib.reload(app)

diagnostic = app.Diagnostiqueur("cpu")
scores = pd.read_csv(C.SCORES_CSV, keep_default_na=False)
test_scores = scores[scores["split"] == "test"]

exemples = {}
exemples["maladie connue"] = test_scores[test_scores["is_known"]]["path"].iloc[0]
exemples["maladie inconnue M"] = test_scores[test_scores["label"] == C.UNKNOWN_CLASS]["path"].iloc[0]
exemples["photo de main"] = scores[scores["category"] == "mains"]["path"].iloc[0]

for nom in exemples:
    chemin = exemples[nom]
    texte, _, _ = diagnostic.predict(Image.open(C.resolve(chemin)))
    print("[" + nom + "]", chemin)
    print(texte)
    print()

[maladie connue] raw/Tomato___Bacterial_spot/00416648-be6e-4bd4-bc8d-82f43f8a7240___GCREC_Bact.Sp 3110.JPG
### Bacterial_spot
Confiance 99.6% (erreur de reconstruction 0.0036). Diagnostic indicatif, à confirmer par un agent agricole.

[maladie inconnue M] raw/Tomato___Tomato_mosaic_virus/046a2559-23b4-4018-a3db-337347ceb391___PSU_CG 2321.JPG
### INCONNU
Le système ne reconnaît pas cette image avec assez de confiance. Ne pas traiter sur cette base : consulter un agent agricole.

Raison : confiance du classifieur 0.67 < 0.93.

[photo de main] ood/photos/mains/WhatsApp Image 2026-10-26 at 11.46.44 (1).jpeg
### INCONNU
Le système ne reconnaît pas cette image avec assez de confiance. Ne pas traiter sur cette base : consulter un agent agricole.

Raison : image atypique (erreur AE 0.0297 > 0.0170).



- Lance l'interface Gradio. Elle s'affiche sous la cellule et à l'adresse indiquée (http://127.0.0.1:7860).
- On dépose une photo, on choisit la règle de rejet, et on obtient le diagnostic, les 3 classes les plus probables et la carte Grad-CAM.

In [12]:
if not C.SMOKE:
    if torch.cuda.is_available():
        demo = app.build("cuda")
    else:
        demo = app.build("cpu")
    demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
